# Scraping: récupération de la page d'accueil de la RTBF

Dans ce notebook, nous créons un robot qui va ouvrir la page d'accueil du site du journal [CAMille](https://max.de.wilde.web.ulb.be/camille/) et récupérer le liens vers les fichiers PDF et les télécharge dans le dossier data/camille

## Imports

In [3]:
import time 
from pathlib import Path
from urllib.parse import urljoin, urlparse

import requests
from bs4 import BeautifulSoup


## Récupération de la liste des fichiers PDF



In [4]:
page_url = "https://max.de.wilde.web.ulb.be/camille/"
headers = {
    "User-Agent": "Mozilla/5.0 (compatible; CAMilleCorpusScraper/1.0)"
}

response = requests.get(page_url, headers=headers, timeout=30)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

pdf_links = list(dict.fromkeys(
    urljoin(page_url, link["href"])
    for link in soup.find_all("a", href=True)
    if ".pdf" in link["href"].lower()
))

print(f"Nombre de fichiers PDF trouvés : {len(pdf_links)}")

Nombre de fichiers PDF trouvés : 51


In [5]:
pdf_links[:10]

['https://max.de.wilde.web.ulb.be/camille/KB_JB230_1892-08-07_01-0003.pdf',
 'https://max.de.wilde.web.ulb.be/camille/KB_JB427_1920-01-10_01-00004.pdf',
 'https://max.de.wilde.web.ulb.be/camille/KB_JB555_1836-02-08_01-00002.pdf',
 'https://max.de.wilde.web.ulb.be/camille/KB_JB638_1860-05-21_01-00002.pdf',
 'https://max.de.wilde.web.ulb.be/camille/KB_JB773_1918-11-30_01-00002.pdf',
 'https://max.de.wilde.web.ulb.be/camille/KB_JB838_1887-12-28_01-00003.pdf',
 'https://max.de.wilde.web.ulb.be/camille/KB_JB230_1903-10-16_01-0002.pdf',
 'https://max.de.wilde.web.ulb.be/camille/KB_JB427_1933-01-04_01-00003.pdf',
 'https://max.de.wilde.web.ulb.be/camille/KB_JB555_1899-01-19_01-00003.pdf',
 'https://max.de.wilde.web.ulb.be/camille/KB_JB638_1902-12-20_01-00002.pdf']

## Vérification du nombre de fichiers et préparation du dossier 

In [6]:
expected_pdf_count = 51

if len(pdf_links) != expected_pdf_count:
    raise ValueError(
        f"Le TP attend {expected_pdf_count} fichiers PDF, "
        f"mais {len(pdf_links)} liens ont été trouvés."
    )

print("Le nombre de liens correspond aux 51 fichiers attendus.")

current_dir = Path.cwd().resolve()
repo_root = next(
    (path for path in (current_dir, *current_dir.parents)
     if (path / ".git").exists()),
    current_dir,
)

output_dir = repo_root / "data" / "camille"
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Les PDF seront enregistrés dans : {output_dir}")

Le nombre de liens correspond aux 51 fichiers attendus.
Les PDF seront enregistrés dans : /Users/krisngajeu/tac/data/camille


## Téléchargement des fichiers PDF

In [7]:
downloaded = 0
skipped = 0

for url in pdf_links:
    filename = Path(urlparse(url).path).name
    destination = output_dir / filename
    temporary_file = output_dir / f"{filename}.part"

    if destination.exists() and destination.stat().st_size > 0:
        skipped += 1
        continue

    with requests.get(
        url, headers=headers, stream=True, timeout=60
    ) as pdf_response:
        pdf_response.raise_for_status()
        chunks = pdf_response.iter_content(chunk_size=8192)
        first_chunk = next(chunks, b"")

        if not first_chunk.startswith(b"%PDF"):
            raise ValueError(f"Le contenu reçu ne semble pas être un PDF : {url}")

        with temporary_file.open("wb") as file:
            file.write(first_chunk)
            for chunk in chunks:
                if chunk:
                    file.write(chunk)

    temporary_file.replace(destination)
    downloaded += 1
    time.sleep(0.2)

print(f"Fichiers téléchargés : {downloaded}")
print(f"Fichiers déjà présents : {skipped}")

Fichiers téléchargés : 51
Fichiers déjà présents : 0


## Vérification des fichiers téléchargés 

In [8]:
pdf_files = sorted(output_dir.glob("*.pdf"))

print(f"Nombre de fichiers PDF dans le dossier : {len(pdf_files)}")
print([file.name for file in pdf_files[:10]])

invalid_files = [
    file for file in pdf_files
    if file.stat().st_size == 0 or file.read_bytes()[:4] != b"%PDF"
]

if len(pdf_files) != expected_pdf_count:
    raise ValueError(
        f"Le dossier devrait contenir {expected_pdf_count} PDF, "
        f"mais en contient {len(pdf_files)}."
    )

if invalid_files:
    raise ValueError(f"Certains fichiers ne semblent pas être des PDF : {invalid_files}")

print("Les 51 fichiers PDF sont présents et vérifiés.")

Nombre de fichiers PDF dans le dossier : 51
['KB_JB230_1892-08-07_01-0003.pdf', 'KB_JB230_1903-10-16_01-0002.pdf', 'KB_JB230_1913-07-05_01-0001.pdf', 'KB_JB258_1884-09-03_01-0003.pdf', 'KB_JB258_1894-12-09_01-0003.pdf', 'KB_JB258_1906-01-09_01-0002.pdf', 'KB_JB421_1899-05-15_01-00003.pdf', 'KB_JB421_1926-10-29_01-00002.pdf', 'KB_JB421_1950-04-15_01-00004.pdf', 'KB_JB427_1920-01-10_01-00004.pdf']
Les 51 fichiers PDF sont présents et vérifiés.


## Pour en savoir plus

- Le web scraping avec Python: https://realpython.com/beautiful-soup-web-scraper-python/
- Tutoriel sur les expressions régulières: https://www.w3schools.com/python/python_regex.asp